# Baixando datast do Kaggle
> Bixando dataset cats and dogs

> O kaggle da o comando no site


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

## Analisando nosso dataset

In [ ]:
import os

data_dir = os.path.join(path, "PetImages")

dog_path = os.path.join(data_dir, 'Dog')
cat_path = os.path.join(data_dir, 'Cat')

dog_imgs = os.listdir(dog_path)
cat_imgs = os.listdir(cat_path)

print(f"Cat images: {len(dog_imgs)}")
print(f"Dog images: {len(cat_imgs)}")
print(f"Total images: {len(dog_imgs) + len(cat_imgs)}")

print(f"First 5 cat images: {cat_imgs[:5]}")
print(f"First 5 dog images: {dog_imgs[:5]}")

dog_imgs = [os.path.join(dog_path, img) for img in dog_imgs]
cat_imgs = [os.path.join(cat_path, img) for img in cat_imgs]

print(f"First cat image: {cat_imgs[0]}")
print(f"First dog image: {dog_imgs[0]}")

## Separando dataset

In [ ]:
import tensorflow as tf
from tensorflow import keras

train_ds, tmp_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    image_size=(128,128),
    seed=123,
    subset="both",
    batch_size=32
)

class_names = train_ds.class_names
print(class_names)



In [ ]:
val_ds = tmp_ds.take(100)
test_ds = tmp_ds.skip(100)

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
tmp_ds = tmp_ds.apply(tf.data.experimental.ignore_errors())

# Versões reduzidas
train_ds_fast = train_ds.take(300)
val_ds_fast = val_ds.take(60)
test_ds_fast = test_ds.take(30)

## Data augmentation

In [5]:
from tensorflow.keras import layers, models

data_augmentation = keras.Sequential([
  layers.RandomFlip("horizontal"),            # Espelha da esquerda pra direita
  layers.RandomRotation(0.1),                 # Gira levemente (até ~36 graus)
  layers.RandomZoom(0.3),                     # Zoom in/out leve (10%)
  layers.RandomTranslation(0.3, 0.5),         # Desloca a imagem em até 10% para os lados/cima
  layers.RandomContrast(0.3)                  # Altera o contraste em 10%
])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))

# Pegamos a primeira imagem do nosso primeiro lote de treino
for images, _ in train_ds.take(1):
  primeira_imagem = images[0]

  # Geramos 9 versões mutantes dela!
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    # Aplicamos a mutação (o training=True força ele a aplicar o efeito)
    imagem_mutante = data_augmentation(tf.expand_dims(primeira_imagem, 0), training=True)

    plt.imshow(imagem_mutante[0].numpy().astype("uint8"))
    plt.axis("off")

plt.show()

## Criando o nosso modelo de Rede Neural Convolucional (CNN)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3), # O mesmo tamanho que já usamos antes!
    include_top=False,         # Cortamos a "boca" da rede original
    weights='imagenet'         # Carregamos a inteligência prévia
)

base_model.trainable = False

In [ ]:


model = models.Sequential([
  # Antes
  # layers.Flatten(input_shape=(28,28)) # Camada de achatametno
  # layers.Dense(512, activation=tensorflow.nn.relu), # camadas da RNA

  # Agora
  # layers.Input(shape=(img_height, img_width, 3)), # falando para rede o tamanho do input
  # layers.Rescaling(1/255.0), # rescale dos pixels
  # layers.Conv2D(32, (3, 3), activation='relu', padding='same'), # varre a imagem procurando 32 padrões diferentes
  # layers.MaxPooling2D((2, 2)), # reduz a imagem pela metade guardando só o que importa
  # layers.BatchNormalization(), # mantém os números equilibrados para a IA aprender mais rápido
  # layers.Dropout(0.25), #  desliga os neurônios aleatoriamente para evitar overfitting

  layers.Input(shape=(128, 128, 3)),

  data_augmentation,

  # layers.Rescaling(1./255), # Para outro modelo (de 0 a 1)

  # layers.Conv2D(16, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Conv2D(32, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Conv2D(64, 3, padding='same', activation='relu'),
  # layers.MaxPooling2D(),

  # layers.Flatten(),

  layers.Rescaling(1./127.5, offset=-1), # MobileNetV2 (de -1 a 1)

  base_model, # Mobile NetV2

  layers.GlobalAveragePooling2D(),

  layers.Dense(128, activation='relu'),
  layers.BatchNormalization(),
  layers.Dropout(0.2),

  layers.Dense(2, activation='softmax')
])

model.summary()

## Compilando e treinando o modelo

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


train_stats = model.fit(
  train_ds_fast,
  validation_data=val_ds_fast,
  epochs=1,
  batch_size=32
)

In [ ]:
print(f"Loss final de Treino: {train_stats.history['loss'][-1]}")
print(f"Acurácia final de Treino: {train_stats.history['accuracy'][-1]}")

print(f"Loss final de Validação: {train_stats.history['val_loss'][-1]}")
print(f"Acurácia final de Validação: {train_stats.history['val_accuracy'][-1]}")

test_loss, test_accuracy = model.evaluate(test_ds_fast)

print(f"Loss de teste foi : {test_loss}")
print(f"Acurácia de teste foi : {test_accuracy}")

## Testando com imagens do google

In [ ]:
import numpy as np
img_url = "https://blog-static.petlove.com.br/wp-content/uploads/2019/06/shutterstock_632318627.jpg"

img_path = tf.keras.utils.get_file(origin=img_url)

img = tf.keras.utils.load_img(
    img_path, target_size=(128, 128)
)

plt.imshow(img)
plt.show()

img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)
img_array = img_array / 255.0

predictions = model.predict(img_array)

classe_predita = class_names[np.argmax(predictions)]
confianca = 100 * np.max(predictions)

print(f"\nResultado da IA: {classe_predita} (com {confianca:.2f}% de confiança!)")